# DSPy Prompt Optimization with CrewAI

This notebook demonstrates how to use `DSPyOptimizer` to algorithmically improve a two-agent email-drafting crew's prompt instructions using MIPROv2.

**What you'll learn:**
- How to define a crew and a labeled training set
- How to write an LLM-judge metric
- How to run `DSPyOptimizer.compile()` and inspect the result
- How to compare baseline vs. optimized crew output

**Prerequisites:**
```bash
pip install 'crewai[dspy]'
```

**Environment variables required:**
- `OPENAI_API_KEY` — used by the crew agents
- `ANTHROPIC_API_KEY` — used by the LLM judge metric

In [ ]:
# Imports and environment setup — loads API keys from environment, never hardcoded
import os
import dspy
import anthropic
from crewai import Agent, Task, Crew, LLM
from crewai.optimizers import DSPyOptimizer

# Verify required keys are present before running expensive LLM calls
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your environment"
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"

# Configure DSPy to use the same model as the crew (gpt-4o-mini is cost-efficient for optimization)
dspy.configure(lm=dspy.LM("openai/gpt-4o-mini", temperature=0.0))

print("Environment ready.")

In [ ]:
# Crew definition: researcher finds facts, writer drafts the email
llm = LLM(model="gpt-4o-mini")

researcher = Agent(
    role="researcher",
    goal="Find key facts and relevant context about the given topic for an email",
    backstory="You are a thorough researcher who quickly identifies the most important facts about any topic.",
    llm=llm,
)

writer = Agent(
    role="writer",
    goal="Draft a professional, clear, and concise business email based on the research",
    backstory="You craft professional emails that are clear, appropriately formal, and to the point.",
    llm=llm,
)

research_task = Task(
    description="Research the topic: {topic}. Identify 3-5 key facts relevant for a business email.",
    expected_output="A bullet list of 3-5 key facts about {topic}",
    agent=researcher,
)

write_task = Task(
    description="Using the research findings, draft a professional business email about {topic}.",
    expected_output="A complete business email with Subject, greeting, body, and sign-off",
    agent=writer,
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    verbose=False,
)

print("Crew defined with 2 agents and 2 tasks.")

In [ ]:
# Baseline measurement: run the crew before optimization to establish the starting quality
baseline_output = crew.kickoff(inputs={"topic": "Q1 earnings call"})

print("=== BASELINE OUTPUT ===")
print(str(baseline_output))

In [ ]:
# Metric function: LLM-as-judge that scores email quality (0.0 to 1.0)
# Uses pairwise comparison with position-bias mitigation via order swap
_judge_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

def _pairwise_score(output_a: str, output_b: str) -> float:
    """Ask Claude to rate A vs B on email quality criteria. Returns float in [0.0, 1.0]."""
    prompt = (
        "You are evaluating two business email drafts. Rate Draft A compared to Draft B "
        "on three criteria: (1) clarity, (2) professional tone, (3) conciseness.\n\n"
        f"Draft A:\n{output_a}\n\nDraft B:\n{output_b}\n\n"
        "Respond with only a single decimal number between 0.0 and 1.0, where:\n"
        "  0.0 = Draft A is much worse than B\n"
        "  0.5 = equally good\n"
        "  1.0 = Draft A is much better than B\n"
        "Number only:"
    )
    resp = _judge_client.messages.create(
        model="claude-haiku-4-5-20251001",  # pinned version for reproducibility
        max_tokens=16,
        messages=[{"role": "user", "content": prompt}],
    )
    try:
        return max(0.0, min(1.0, float(resp.content[0].text.strip())))
    except (ValueError, IndexError):
        return 0.5  # neutral fallback if judge gives unparseable output


def email_quality_metric(example: dspy.Example, prediction) -> float:
    """Score the predicted email against the reference output using a pairwise LLM judge."""
    candidate = str(prediction)
    reference = example.output

    # Average forward and reverse scores to eliminate position bias
    forward = _pairwise_score(candidate, reference)    # A=candidate, B=reference
    reverse = 1.0 - _pairwise_score(reference, candidate)  # A=reference, B=candidate → invert
    return (forward + reverse) / 2.0


print("Metric function defined.")

In [ ]:
# Training set: 10 diverse email-drafting examples with realistic reference outputs
# These cover different email types so the optimizer learns generalizable improvements
trainset = [
    dspy.Example(
        inputs={"topic": "Q1 earnings call"},
        output=(
            "Subject: Invitation: Q1 Earnings Call — [Date] at 10:00 AM EST\n\n"
            "Dear Team,\n\n"
            "You are invited to our Q1 earnings call on [Date] at 10:00 AM EST. "
            "We will review revenue performance, operating margins, and forward guidance.\n\n"
            "Please join via the link below. Contact investor.relations@company.com with questions.\n\n"
            "Best regards,\nInvestor Relations"
        ),
    ).with_inputs("topic"),
    dspy.Example(
        inputs={"topic": "product launch announcement"},
        output=(
            "Subject: Announcing [Product Name] — Now Available\n\n"
            "Dear Valued Partner,\n\n"
            "We are excited to announce the launch of [Product Name], "
            "now available for all customers. Key features include improved performance, "
            "enhanced security, and seamless integration.\n\n"
            "Visit our website or contact your account manager for details.\n\n"
            "Sincerely,\nProduct Team"
        ),
    ).with_inputs("topic"),
    dspy.Example(
        inputs={"topic": "office closure for public holiday"},
        output=(
            "Subject: Office Closure — [Holiday Name], [Date]\n\n"
            "Dear Colleagues,\n\n"
            "Please note that our offices will be closed on [Date] in observance of [Holiday Name]. "
            "Normal operations resume on [Next Business Day].\n\n"
            "For urgent matters, please contact the on-call team at oncall@company.com.\n\n"
            "Thank you,\nHR Team"
        ),
    ).with_inputs("topic"),
    dspy.Example(
        inputs={"topic": "new employee onboarding policy update"},
        output=(
            "Subject: Updated Onboarding Policy — Effective [Date]\n\n"
            "Dear Managers,\n\n"
            "We have updated our employee onboarding policy, effective [Date]. "
            "Key changes include a new 30-day check-in process and updated IT provisioning timelines.\n\n"
            "Please review the full policy at [link] and ensure your team is briefed before their start dates.\n\n"
            "Best,\nPeople Operations"
        ),
    ).with_inputs("topic"),
    dspy.Example(
        inputs={"topic": "quarterly business review invitation"},
        output=(
            "Subject: QBR — [Quarter] Business Review, [Date]\n\n"
            "Hi [Name],\n\n"
            "I'd like to invite you to our [Quarter] Business Review on [Date] at [Time]. "
            "We will cover account performance, upcoming initiatives, and strategic priorities.\n\n"
            "Please confirm your attendance by [RSVP Date].\n\n"
            "Looking forward to connecting,\n[Your Name]"
        ),
    ).with_inputs("topic"),
    dspy.Example(
        inputs={"topic": "security incident notification"},
        output=(
            "Subject: Important: Security Incident Notification\n\n"
            "Dear Customer,\n\n"
            "We are writing to inform you of a security incident that may have affected your account. "
            "On [Date], we detected unauthorized access to [affected system]. "
            "We have contained the issue and taken corrective action.\n\n"
            "We recommend changing your password immediately. Contact security@company.com with questions.\n\n"
            "Sincerely,\nSecurity Team"
        ),
    ).with_inputs("topic"),
    dspy.Example(
        inputs={"topic": "vendor contract renewal"},
        output=(
            "Subject: Contract Renewal — [Vendor Name] Agreement Expiring [Date]\n\n"
            "Dear [Vendor Contact],\n\n"
            "Our current agreement with [Vendor Name] expires on [Date]. "
            "We would like to renew for another 12-month term with the current pricing and scope.\n\n"
            "Please send the renewal documents by [Date - 30 days] to allow time for review and signature.\n\n"
            "Thank you,\nProcurement Team"
        ),
    ).with_inputs("topic"),
    dspy.Example(
        inputs={"topic": "remote work policy change"},
        output=(
            "Subject: Updated Remote Work Policy — Effective [Date]\n\n"
            "Dear Team,\n\n"
            "Following our recent review, we are updating our remote work policy effective [Date]. "
            "Employees may work remotely up to 3 days per week, subject to manager approval.\n\n"
            "Full details are available in the Employee Handbook. "
            "Questions? Reach out to your HR business partner.\n\n"
            "Best,\nHR"
        ),
    ).with_inputs("topic"),
    dspy.Example(
        inputs={"topic": "budget approval request"},
        output=(
            "Subject: Budget Approval Request — [Project Name], $[Amount]\n\n"
            "Hi [Approver],\n\n"
            "I am requesting approval for $[Amount] to fund [Project Name] in [Quarter]. "
            "This investment supports [business objective] and is expected to deliver [ROI].\n\n"
            "Full budget breakdown is attached. Please approve or provide feedback by [Date].\n\n"
            "Thank you,\n[Your Name]"
        ),
    ).with_inputs("topic"),
    dspy.Example(
        inputs={"topic": "team restructuring announcement"},
        output=(
            "Subject: Team Restructuring — Changes Effective [Date]\n\n"
            "Dear Team,\n\n"
            "I want to share some important changes to our team structure, effective [Date]. "
            "[Name] will take on the role of [New Title], and [Team/Function] will now report to [Manager].\n\n"
            "These changes position us better to [strategic goal]. "
            "A team meeting is scheduled for [Date] to discuss questions.\n\n"
            "Thank you for your continued dedication,\n[Manager Name]"
        ),
    ).with_inputs("topic"),
]

print(f"Training set: {len(trainset)} examples covering diverse email types.")

In [ ]:
# Optimizer construction and compile — this is the main optimization step
# num_trials=20 means MIPROv2 will explore 20 candidate instruction rewrites
optimizer = DSPyOptimizer(
    crew=crew,
    metric=email_quality_metric,
    algorithm="MIPROv2",
)

print("Starting optimization — this will take several minutes...")
result = optimizer.compile(trainset=trainset, num_trials=20)
print("Optimization complete.")

In [ ]:
# Inspect the result: score improvement and optimized agent instructions
print(f"=== OPTIMIZATION RESULT ===")
print(f"Version ID:       {result.version_id}")
print(f"Baseline score:   {result.baseline_score:.3f}")
print(f"Optimized score:  {result.optimized_score:.3f}")
print(f"Score delta:      {result.score_delta:+.3f}")
print(f"Trials run:       {result.num_trials}")
print()

for role, instructions in result.optimized_instructions.items():
    print(f"--- Agent: {role} ---")
    if instructions.backstory:
        print(f"Optimized backstory:\n{instructions.backstory}\n")

In [ ]:
# Run the optimized crew and compare with baseline output
optimized_output = result.crew.kickoff(inputs={"topic": "Q1 earnings call"})

print("=== BASELINE OUTPUT ===")
print(str(baseline_output))
print()
print("=== OPTIMIZED OUTPUT ===")
print(str(optimized_output))
print()
print(f"Score improvement: {result.score_delta:+.3f}")